# Metro Ridership Update (legacy)

**Status: Partially obsolete**

This notebook merged scraped data with the historical JSON. The scraper-based workflow is
no longer used — see `metro_data_ridership_update.ipynb` for the current CSV-based pipeline.

The `convert_to_nested_json` helper and the data comparison cells at the bottom may still
be useful for debugging. The scraper import (`import metro_ridership_scraper as scraper`)
and cells that call `scraper.*` functions will fail in this repo.

In [ ]:
import pandas as pd
import metro_ridership_scraper as scraper
import json
import numpy as np

pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 1000)

In [ ]:
def convert_to_nested_json(df):
    print('converting data to nested json')
    df_json = df.groupby(
        ['year', 'month', 'line_name']
    )[[
        'est_wkday_ridership',
        'est_sat_ridership',
        'est_sun_ridership'
    ]].apply(
        lambda x: x.to_dict('records')
    ).reset_index(
        name='data'
    )
    df_json['new'] = df_json.data.str[0]
    json = df_json.groupby(
            ['year', 'month']
        )[[
            'line_name','new'
        ]].apply(
            lambda x: x.set_index('line_name')['new'].to_dict()
        ).reset_index(
            name='data'
    ).groupby(
            ['year']
        )[[
            'month','data'
        ]].apply(
            lambda x: x.set_index('month')['data'].to_dict()
    ).to_json(indent = 2)

    return json

## get new data

In [ ]:
new_df = pd.read_csv('Monthly_Riders.csv')
display(new_df.head())

In [ ]:
new_df[
    new_df.duplicated(subset=[
        'Year', 'Month', 'Line','DayType'
    ], keep=False)&
    (new_df.Line==2)
].sort_values(
    by=['Year', 'Month', 'Line','DayType'],
    ascending = False
).head(10)

In [ ]:
# df = new_df[
#     (new_df.Year==2014)&
#     (new_df.Month==12)&
#     (new_df.Line==2)&
#     (new_df.DayType=='SU')
# ].copy()
# df

In [ ]:
# recalculate average ridership accomodating for shakeups
# it looks like metro people round these numbers up (hence +0.5)
wm = lambda x: int(np.average(x+0.5, weights=new_df.loc[x.index, 'Days']))
adjusted_df = new_df.groupby(
    ['Year','Month', 'Line', 'DayType', 'Provider', 'Mode']
).agg(riders_weighted=('Riders', wm)).reset_index(drop=False)
adjusted_df.head(1)

In [ ]:
adjusted_full_df = adjusted_df.pivot_table(
    values=['riders_weighted'],
    index=['Year','Month','Line', 'Provider', 'Mode'],
    columns=['DayType']
).reset_index(drop=False, col_level=1)
adjusted_full_df.head()

In [ ]:
adjusted_full_df.columns = adjusted_full_df.columns.droplevel()
adjusted_full_df.rename(
    columns = {
        'Year': 'year',
        'Month': 'month', 
        'Line': 'line_name',
        'DX':  'est_wkday_ridership', 
        'SA':  'est_sat_ridership', 
        'SU':  'est_sun_ridership'
    },
    inplace = True
)
adjusted_full_df = adjusted_full_df.rename_axis(None, axis=1)
adjusted_full_df.line_name = adjusted_full_df.line_name.astype(str)

In [ ]:
adjusted_full_df.head()

In [ ]:
metro_df = adjusted_full_df[[
    'year', 'month', 'line_name',
  'est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership'
]].copy()
display(metro_df.head())
metro_df.to_csv(f'metro_data_adjusted.csv', index=False)

In [ ]:
json_nested = convert_to_nested_json(metro_df)
with open(f'metro_ridership_nested_current.json', 'w') as f:
    f.write(json_nested)

json_flat = metro_df.to_json(orient = 'records', indent = 2)
with open(f'metro_ridership_flat_current.json', 'w') as f:
     f.write(json_flat)

## compare with existing data

In [ ]:
with open(f'metro_ridership_current.json', 'r') as f:
    current_json = json.load(f)

In [ ]:
current_df = pd.DataFrame(
    [
        (
            year, month, line,
            info['est_wkday_ridership'],
            info['est_sat_ridership'],
            info['est_sun_ridership']
        )
        for year, months in current_json.items()
        for month, lines in months.items()
        for line, info in lines.items()
    ],
    columns=[
    'year', 'month', 'line_name',
    'est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership'
    ]
)
display(current_df.head())

In [ ]:
current_df[
    (current_df.year=='2017')&
    (current_df.month=='12')&
    (current_df.line_name=='2')
]

In [ ]:
new_df[
    (new_df.Year==2023)&
    (new_df.Month==12)&
    (new_df.Line==2)
]

In [ ]:
final_df = pd.concat(
    [format_df, current_df]
).drop_duplicates(
    subset = ['year', 'month', 'line_name'],
    keep = 'first'
).reset_index(drop=True)

In [ ]:
if final_df.shape[0]>current_df.shape[0]:
    final_json = scraper.convert_df_to_json(final_df)
    with open(f'metro_ridership_current.json', 'w') as f:
        f.write(final_json)
    print('update saved')
else:
    print('no updates')

## compare data from metro with scraped data

In [ ]:
df_scraped = pd.read_csv(f'metro_ridership_scraped_2024_05_10.csv')
df_scraped = df_scraped[df_scraped.line_name!='All'].copy()

In [ ]:
compare_df = metro_df.merge(
    dfs_scraped, 
    on = ['year', 'month', 'line_name'],
    how = 'outer',
    suffixes = ('_metro', '_scraped')
)

compare_df['wkday_delta'] = compare_df.est_wkday_ridership_metro - \
    compare_df.est_wkday_ridership_scraped
compare_df['sat_delta'] = compare_df.est_sat_ridership_metro - \
    compare_df.est_sat_ridership_scraped
compare_df['sun_delta'] = compare_df.est_sun_ridership_metro - \
    compare_df.est_sun_ridership_scraped

compare_df[
    (
        (
            (compare_df.est_wkday_ridership_metro!=
             compare_df.est_wkday_ridership_scraped)&
            compare_df.est_wkday_ridership_metro.notnull()&
            compare_df.est_wkday_ridership_scraped.notnull()
        )|
        (
            (compare_df.est_sat_ridership_metro!=
             compare_df.est_sat_ridership_scraped)&
            compare_df.est_sat_ridership_metro.notnull()&
            compare_df.est_sat_ridership_scraped.notnull()
        )|
        (
            (compare_df.est_sun_ridership_metro!=
             compare_df.est_sun_ridership_scraped)&
            compare_df.est_sun_ridership_metro.notnull()&
            compare_df.est_sun_ridership_scraped.notnull()
        )
    )&(
        (compare_df.wkday_delta!=0)|
        (compare_df.sat_delta!=0)|
        (compare_df.sun_delta!=0)
    )
]

In [ ]:
compare_df[
    compare_df.est_wkday_ridership_metro.isnull()&
    compare_df.est_sat_ridership_metro.isnull()&
    compare_df.est_sun_ridership_metro.isnull()
]